# 🌾 SpectraFarm — MP / UP Ground-Truth Crop Label Collector

## Purpose

This notebook lets you **manually collect real, geo-tagged crop ground-truth points** for Madhya Pradesh and/or Uttar Pradesh — the two primary states our SpectraFarm (AgriN) system targets.

### Why does this exist?

Our Random Forest crop classifier is currently trained on a mix of **synthetic features** and **CropHarvest** labels that cover India broadly but are thin for MP/UP specifically. To improve classification accuracy in our pilot regions, we need **localized, field-verified labels** tied to specific districts and seasons.

### What it produces

A CSV file (`data/ground_truth/mp_up_manual_labels.csv`) with the **exact same 17-feature + metadata schema** as `training_features.csv`, so it can be merged directly into the existing `load_merged_ground_truth()` pipeline without any schema mapping.

### Two ways to label

1. **Click-to-tag** — Click directly on the satellite map to drop a point, then tag it with crop type / district / date.
2. **KML/GeoJSON import** — Upload polygon files (e.g. from Google Earth Pro field surveys) and attach labels via an editable table.

---

**This notebook only collects and exports labeled data — it does NOT train any model.**

---
## 1. Setup — Install Dependencies & Authenticate GEE

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 1. Install required packages (Colab-friendly)
# ═══════════════════════════════════════════════════════════════════════════

import subprocess, sys

def _install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

_install("geemap")
_install("earthengine-api")
_install("geopandas")
_install("fiona")
_install("lxml")
_install("ipywidgets")

print("✅ All packages installed.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 2. Imports
# ═══════════════════════════════════════════════════════════════════════════

import ee
import geemap
import pandas as pd
import geopandas as gpd
import numpy as np
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from datetime import datetime, date
from pathlib import Path
import json, os, io, warnings
warnings.filterwarnings('ignore')

print("✅ Imports complete.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 3. Authenticate & Initialize Google Earth Engine
# ═══════════════════════════════════════════════════════════════════════════

try:
    ee.Initialize(project='agrin-506618')
    print("✅ Earth Engine initialized (existing credentials).")
except Exception:
    ee.Authenticate()
    ee.Initialize(project='agrin-506618')
    print("✅ Earth Engine authenticated and initialized.")

---
## 2. State Selection & Map Scoping

### How state-boundary filtering works

We load official state boundaries from the **FAO GAUL Level-1** administrative dataset on Earth Engine (`FAO/GAUL/2015/level1`). When you select a state:

1. The boundary geometry is extracted and used to **clip** the Sentinel-2 composite — imagery outside the exact state polygon is transparent.
2. The state's bounding box (with 0.2° padding) is set as the map's **`max_bounds`** — you physically **cannot pan** outside the state.
3. The map automatically **centers and zooms** to the selected state's extent.
4. Any click-to-tag point that falls **outside** the exact polygon (not just the bounding box) is rejected with a warning.
5. KML/GeoJSON polygons are validated against the exact polygon — centroids outside are flagged.

You can re-run the cells below at any time to switch states without restarting the notebook. The `max_bounds` and clip mask are recomputed automatically.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# State Selection — re-run this cell to switch states
# ═══════════════════════════════════════════════════════════════════════════

# ──── Configuration ────────────────────────────────────────────────────────
SELECTED_STATE = "Madhya Pradesh"  # Options: "Madhya Pradesh", "Uttar Pradesh", "Both"
SEASON = "Rabi"                    # Options: "Rabi" (Nov-Mar), "Kharif" (Jun-Oct)
SEASON_YEAR = 2024                 # Year for the composite
# ──────────────────────────────────────────────────────────────────────────

# Date ranges by season
if SEASON == "Rabi":
    START_DATE = f"{SEASON_YEAR - 1}-11-01"
    END_DATE   = f"{SEASON_YEAR}-03-31"
elif SEASON == "Kharif":
    START_DATE = f"{SEASON_YEAR}-06-01"
    END_DATE   = f"{SEASON_YEAR}-10-31"
else:
    raise ValueError(f"Unknown season: {SEASON}. Use 'Rabi' or 'Kharif'.")

print(f"📅 Season: {SEASON} {SEASON_YEAR}  →  {START_DATE} to {END_DATE}")

# Load state boundaries from FAO GAUL Level 1
gaul = ee.FeatureCollection("FAO/GAUL/2015/level1")

if SELECTED_STATE == "Both":
    state_fc = gaul.filter(
        ee.Filter.Or(
            ee.Filter.eq("ADM1_NAME", "Madhya Pradesh"),
            ee.Filter.eq("ADM1_NAME", "Uttar Pradesh")
        )
    )
    state_label = "Madhya Pradesh & Uttar Pradesh"
else:
    state_fc = gaul.filter(ee.Filter.eq("ADM1_NAME", SELECTED_STATE))
    state_label = SELECTED_STATE

state_geom = state_fc.geometry()
print(f"🗺️  State boundary loaded: {state_label}")

# ──── Compute bounding box for map pan limits ────────────────────────────
bbox_coords = state_geom.bounds().getInfo()['coordinates'][0]
lons = [c[0] for c in bbox_coords]
lats = [c[1] for c in bbox_coords]
PAD = 0.2  # degrees of padding so edges aren't flush
state_south = min(lats) - PAD
state_north = max(lats) + PAD
state_west  = min(lons) - PAD
state_east  = max(lons) + PAD
print(f"📐 Pan bounds: [{state_south:.2f}°N, {state_west:.2f}°E] → [{state_north:.2f}°N, {state_east:.2f}°E]")

# ──── Build cloud-masked Sentinel-2 composite, clipped to exact state polygon
def _mask_s2_clouds(img):
    qa = img.select('QA60')
    cloud_mask = qa.bitwiseAnd(1 << 10).eq(0).And(qa.bitwiseAnd(1 << 11).eq(0))
    return img.updateMask(cloud_mask).divide(10000)

s2_composite = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterDate(START_DATE, END_DATE)
    .filterBounds(state_geom)
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 30))
    .map(_mask_s2_clouds)
    .median()
    .clip(state_geom)  # Clip to exact polygon — neighboring states are transparent
)

print(f"🛰️  Sentinel-2 composite built ({START_DATE} → {END_DATE}), clipped to {state_label}.")
print("   Re-run this cell to change state or season.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Render the interactive map (locked to selected state bounds)
# ═══════════════════════════════════════════════════════════════════════════

Map = geemap.Map()
Map.centerObject(state_fc, zoom=7)

# Lock panning to the state bounding box (with padding)
# geemap.Map wraps ipyleaflet.Map — max_bounds prevents dragging outside
Map.max_bounds = [[state_south, state_west], [state_north, state_east]]
Map.min_zoom = 6   # Prevent zooming out so far the bounds don't matter

# State outline — thick, bright cyan border for clear visibility
Map.addLayer(
    state_fc.style(color='00FFFF', width=3, fillColor='00000011'),
    {},
    f'{state_label} Boundary'
)

# True-color Sentinel-2 composite (clipped to exact state polygon)
vis_params = {
    'bands': ['B4', 'B3', 'B2'],
    'min': 0.0,
    'max': 0.3,
    'gamma': 1.3
}
Map.addLayer(s2_composite, vis_params, f'Sentinel-2 {SEASON} {SEASON_YEAR}')

# False-color (NIR-R-G) for vegetation emphasis
nir_vis = {
    'bands': ['B8', 'B4', 'B3'],
    'min': 0.0,
    'max': 0.4,
    'gamma': 1.2
}
Map.addLayer(s2_composite, nir_vis, f'NIR False-Color {SEASON}', shown=False)

print(f"🗺️  Map ready — locked to {state_label}. You cannot pan outside the state.")
print(f"   Zoom in/out works freely within the state bounds.")
Map

---
## 3. Click-to-Tag Labeling

### How to use

1. **Click** anywhere on the map above to drop a marker.
2. A small form appears below the map — select the **crop type**, type the **district name**, and pick a **date**.
3. Click **"Save Point"** to add it to the DataFrame.
4. If your click falls **outside** the selected state's exact polygon boundary (not just the bounding box), it will be rejected with a warning.
5. Review saved points in the live table below the form.

The 10 crop classes match the project's `CropType` enum: Wheat, Rice, Mustard, Soybean, Cotton, Sugarcane, Potato, Lentil, Maize, Gram.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Click-to-Tag Labeling Widget
# ═══════════════════════════════════════════════════════════════════════════

# Storage for collected points
collected_points = []

# 10 crop classes matching CropType enum in src/data/schemas.py
CROP_CLASSES = [
    "Wheat", "Rice", "Mustard", "Soybean", "Cotton",
    "Sugarcane", "Potato", "Lentil", "Maize", "Gram"
]

# ──── Form widgets ─────────────────────────────────────────────────────────
lbl_lat = widgets.FloatText(description='Lat:', disabled=True, layout=widgets.Layout(width='280px'))
lbl_lon = widgets.FloatText(description='Lon:', disabled=True, layout=widgets.Layout(width='280px'))
crop_dropdown = widgets.Dropdown(
    options=CROP_CLASSES,
    value='Wheat',
    description='Crop:',
    layout=widgets.Layout(width='280px')
)
district_text = widgets.Text(
    description='District:',
    placeholder='e.g. Sehore, Lucknow',
    layout=widgets.Layout(width='280px')
)
date_picker = widgets.DatePicker(
    description='Date:',
    value=date.today(),
    layout=widgets.Layout(width='280px')
)
save_btn = widgets.Button(
    description='💾 Save Point',
    button_style='success',
    layout=widgets.Layout(width='280px')
)
status_output = widgets.Output(layout=widgets.Layout(width='100%'))
table_output = widgets.Output(layout=widgets.Layout(width='100%'))

form_box = widgets.VBox([
    widgets.HTML('<h4 style="color:#0284c7; margin:6px 0;">📍 Tag Clicked Point</h4>'),
    widgets.HBox([lbl_lat, lbl_lon]),
    widgets.HBox([crop_dropdown, district_text]),
    widgets.HBox([date_picker, save_btn]),
    status_output,
    widgets.HTML('<h4 style="color:#059669; margin:10px 0 4px;">📋 Collected Points</h4>'),
    table_output,
])


def _refresh_table():
    """Update the displayed table of collected points."""
    with table_output:
        clear_output(wait=True)
        if collected_points:
            df = pd.DataFrame(collected_points)
            display(df.style.set_caption(f'{len(df)} points collected'))
        else:
            print('No points collected yet. Click on the map to start.')


def _on_map_click(event, **kwargs):
    """Handle map click — populate lat/lon fields."""
    if event == 'click':
        coords = kwargs.get('coordinates')
        if coords:
            lat, lon = coords[0], coords[1]
            lbl_lat.value = round(lat, 6)
            lbl_lon.value = round(lon, 6)
            with status_output:
                clear_output(wait=True)
                print(f'📍 Clicked: {lat:.6f}°N, {lon:.6f}°E — fill the form and click Save.')


def _on_save_click(b):
    """Validate against exact state polygon and save the tagged point."""
    lat = lbl_lat.value
    lon = lbl_lon.value

    if lat == 0.0 and lon == 0.0:
        with status_output:
            clear_output(wait=True)
            print('⚠️  No point selected. Click on the map first.')
        return

    # Validate against exact state polygon (not just bounding box)
    point = ee.Geometry.Point([lon, lat])
    inside = state_geom.contains(point).getInfo()

    if not inside:
        with status_output:
            clear_output(wait=True)
            print(f'❌ REJECTED — Point ({lat:.4f}°N, {lon:.4f}°E) is OUTSIDE the exact {state_label} polygon.')
            print('   The point may be within the bounding box but outside the actual state shape.')
            print('   Only points within the exact state boundary are accepted.')
        return

    if not district_text.value.strip():
        with status_output:
            clear_output(wait=True)
            print('⚠️  Please enter a district name.')
        return

    # Determine state name
    if SELECTED_STATE == "Both":
        mp_geom = gaul.filter(ee.Filter.eq("ADM1_NAME", "Madhya Pradesh")).geometry()
        in_mp = mp_geom.contains(point).getInfo()
        pt_state = "Madhya Pradesh" if in_mp else "Uttar Pradesh"
    else:
        pt_state = SELECTED_STATE

    record = {
        'lat': round(lat, 6),
        'lon': round(lon, 6),
        'crop': crop_dropdown.value,
        'district': district_text.value.strip(),
        'date': str(date_picker.value),
        'state': pt_state,
        'source': 'manual_click',
    }
    collected_points.append(record)

    Map.add_marker(location=[lat, lon], popup=f"{record['crop']} — {record['district']}")

    with status_output:
        clear_output(wait=True)
        print(f'✅ Saved: {record["crop"]} in {record["district"]}, {pt_state} ({lat:.4f}°N, {lon:.4f}°E)')

    _refresh_table()


Map.on_interaction(_on_map_click)
save_btn.on_click(_on_save_click)

_refresh_table()
display(form_box)

---
## 4. Optional — KML / GeoJSON Import

### How to use

If you have field polygons drawn in **Google Earth Pro** or another GIS tool:

1. Export your polygons as a `.kml` or `.geojson` file.
2. Upload the file using the button below.
3. For each polygon, the centroid is computed and validated against the **exact state polygon** (not just the bounding box).
4. An editable table appears where you can assign **crop type**, **district**, and **date** to each polygon.
5. Click **"Merge into Dataset"** to combine with the click-tagged points.

Points outside the selected state polygon are flagged and excluded.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# KML / GeoJSON Import
# ═══════════════════════════════════════════════════════════════════════════

kml_points = []

upload_widget = widgets.FileUpload(
    accept='.kml,.geojson,.json',
    multiple=False,
    description='Upload KML/GeoJSON'
)
kml_output = widgets.Output()
kml_table_output = widgets.Output()
merge_btn = widgets.Button(
    description='🔗 Merge into Dataset',
    button_style='primary',
    layout=widgets.Layout(width='240px')
)


def _parse_upload(change):
    """Parse uploaded KML or GeoJSON file."""
    global kml_points
    kml_points = []

    with kml_output:
        clear_output(wait=True)

        if not upload_widget.value:
            print('No file uploaded.')
            return

        uploaded = list(upload_widget.value.values())[0] if isinstance(upload_widget.value, dict) else upload_widget.value[0]
        content = uploaded['content'] if isinstance(uploaded, dict) else uploaded.content
        name = uploaded.get('name', 'upload') if isinstance(uploaded, dict) else getattr(uploaded, 'name', 'upload')

        try:
            if name.lower().endswith('.kml'):
                import fiona
                fiona.drvsupport.supported_drivers['KML'] = 'r'
                tmp_path = '/tmp/_upload.kml'
                with open(tmp_path, 'wb') as f:
                    f.write(content)
                gdf = gpd.read_file(tmp_path, driver='KML')
            else:
                gdf = gpd.read_file(io.BytesIO(content))

            print(f'📂 Parsed {len(gdf)} feature(s) from {name}')

            # Get state boundary as shapely for fast local checks
            state_geo_info = state_fc.getInfo()
            state_shapely = gpd.GeoDataFrame.from_features(state_geo_info['features']).unary_union

            accepted = 0
            rejected = 0
            for idx, row in gdf.iterrows():
                geom = row.geometry
                if geom is None:
                    continue
                centroid = geom.centroid
                lat, lon = centroid.y, centroid.x

                from shapely.geometry import Point as ShapelyPoint
                if not state_shapely.contains(ShapelyPoint(lon, lat)):
                    rejected += 1
                    continue

                feature_name = row.get('Name', row.get('name', f'polygon_{idx}'))
                kml_points.append({
                    'id': f'KML_{idx:04d}',
                    'name': str(feature_name),
                    'lat': round(lat, 6),
                    'lon': round(lon, 6),
                    'crop': 'Wheat',
                    'district': '',
                    'date': str(date.today()),
                })
                accepted += 1

            print(f'  ✅ {accepted} centroid(s) inside {state_label}')
            if rejected > 0:
                print(f'  ❌ {rejected} centroid(s) OUTSIDE {state_label} — excluded')

        except Exception as e:
            print(f'❌ Error parsing file: {e}')
            return

    with kml_table_output:
        clear_output(wait=True)
        if kml_points:
            kml_df = pd.DataFrame(kml_points)
            print('Edit crop/district/date below, then click "Merge into Dataset":')
            display(kml_df)
        else:
            print('No valid points inside the state boundary.')


def _on_merge_click(b):
    """Merge KML points into the main collected_points list."""
    if not kml_points:
        with kml_output:
            clear_output(wait=True)
            print('⚠️  No KML points to merge. Upload a file first.')
        return

    for pt in kml_points:
        if SELECTED_STATE == "Both":
            mp_geom_info = gaul.filter(ee.Filter.eq("ADM1_NAME", "Madhya Pradesh")).geometry()
            in_mp = mp_geom_info.contains(ee.Geometry.Point([pt['lon'], pt['lat']])).getInfo()
            pt_state = "Madhya Pradesh" if in_mp else "Uttar Pradesh"
        else:
            pt_state = SELECTED_STATE

        collected_points.append({
            'lat': pt['lat'],
            'lon': pt['lon'],
            'crop': pt['crop'],
            'district': pt['district'],
            'date': pt['date'],
            'state': pt_state,
            'source': 'manual_kml',
        })

    with kml_output:
        clear_output(wait=True)
        print(f'✅ Merged {len(kml_points)} KML points into dataset. Total: {len(collected_points)} points.')

    _refresh_table()


upload_widget.observe(_parse_upload, names='value')
merge_btn.on_click(_on_merge_click)

display(widgets.VBox([
    widgets.HTML('<h4 style="color:#7c3aed;">📂 Import KML / GeoJSON Polygons</h4>'),
    upload_widget,
    kml_output,
    kml_table_output,
    merge_btn,
]))

---
## 5. Feature Extraction (Sentinel-2 + Sentinel-1 via Earth Engine)

### What this does

For every tagged point, we pull **Sentinel-2 (optical)** and **Sentinel-1 (SAR)** imagery from Earth Engine over a ±45-day window around the labeled date, then compute the **same 17 features** used by our Random Forest classifier:

| # | Feature | Source | Description |
|---|---------|--------|-------------|
| 1 | `ndvi_mean` | S2 | Mean NDVI over the time window |
| 2 | `ndvi_min` | S2 | Minimum NDVI |
| 3 | `ndvi_max` | S2 | Maximum NDVI |
| 4 | `ndvi_std` | S2 | NDVI standard deviation |
| 5 | `ndvi_range` | S2 | NDVI max − min |
| 6 | `ndvi_slope` | S2 | Linear trend slope |
| 7 | `ndwi_mean` | S2 | Mean NDWI (moisture index) |
| 8 | `vv_mean` | S1 | Mean VV backscatter (dB) |
| 9 | `vv_std` | S1 | VV standard deviation |
| 10 | `vh_mean` | S1 | Mean VH backscatter (dB) |
| 11 | `vh_std` | S1 | VH standard deviation |
| 12 | `vh_vv_ratio` | S1 | Mean VH / VV ratio |
| 13 | `red_mean` | S2 | Mean Red reflectance |
| 14 | `green_mean` | S2 | Mean Green reflectance |
| 15 | `blue_mean` | S2 | Mean Blue reflectance |
| 16 | `nir_mean` | S2 | Mean NIR reflectance |
| 17 | `swir1_mean` | S2 | Mean SWIR-1 reflectance |

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 17-Feature Extraction from GEE (mirrors src/features/feature_extraction.py)
# ═══════════════════════════════════════════════════════════════════════════

def extract_17_features(lat, lon, center_date, buffer_m=500):
    """
    Extract 17 spectral/SAR features for a single point from Earth Engine.
    Mirrors the exact schema in src/features/feature_extraction.py.
    """
    from datetime import datetime as _dt, timedelta

    center_dt = _dt.strptime(center_date, '%Y-%m-%d')
    start_dt = center_dt - timedelta(days=45)
    end_dt   = center_dt + timedelta(days=45)
    start_str = start_dt.strftime('%Y-%m-%d')
    end_str   = end_dt.strftime('%Y-%m-%d')

    point = ee.Geometry.Point([lon, lat])
    aoi = point.buffer(buffer_m)

    features = {}

    # ── Sentinel-2 Optical ────────────────────────────────────────────────
    try:
        s2 = (
            ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
            .filterDate(start_str, end_str)
            .filterBounds(aoi)
            .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 30))
            .map(_mask_s2_clouds)
        )

        def _add_indices(img):
            ndvi = img.normalizedDifference(['B8', 'B4']).rename('NDVI')
            ndwi = img.normalizedDifference(['B8', 'B11']).rename('NDWI')
            return img.addBands([ndvi, ndwi])

        s2 = s2.map(_add_indices)
        n_images = s2.size().getInfo()

        if n_images > 0:
            def _extract_ndvi(img):
                val = img.select('NDVI').reduceRegion(
                    reducer=ee.Reducer.mean(), geometry=aoi, scale=10, maxPixels=1e6
                )
                return ee.Feature(None, {'ndvi': val.get('NDVI'), 'ts': img.get('system:time_start')})

            ndvi_fc = s2.map(_extract_ndvi).getInfo()
            ndvi_vals = []
            for f in ndvi_fc.get('features', []):
                v = f['properties'].get('ndvi')
                if v is not None:
                    ndvi_vals.append(float(v))

            if ndvi_vals:
                ndvi_arr = np.array(ndvi_vals)
                features['ndvi_mean']  = float(np.mean(ndvi_arr))
                features['ndvi_min']   = float(np.min(ndvi_arr))
                features['ndvi_max']   = float(np.max(ndvi_arr))
                features['ndvi_std']   = float(np.std(ndvi_arr))
                features['ndvi_range'] = float(np.max(ndvi_arr) - np.min(ndvi_arr))
                if len(ndvi_arr) >= 3:
                    x = np.arange(len(ndvi_arr), dtype=float)
                    features['ndvi_slope'] = float(np.polyfit(x, ndvi_arr, 1)[0])
                else:
                    features['ndvi_slope'] = 0.0

            composite = s2.median()
            band_stats = composite.select(['B2', 'B3', 'B4', 'B8', 'B11', 'NDWI']).reduceRegion(
                reducer=ee.Reducer.mean(), geometry=aoi, scale=10, maxPixels=1e6
            ).getInfo()

            features['blue_mean']  = float(band_stats.get('B2', 0) or 0)
            features['green_mean'] = float(band_stats.get('B3', 0) or 0)
            features['red_mean']   = float(band_stats.get('B4', 0) or 0)
            features['nir_mean']   = float(band_stats.get('B8', 0) or 0)
            features['swir1_mean'] = float(band_stats.get('B11', 0) or 0)
            features['ndwi_mean']  = float(band_stats.get('NDWI', 0) or 0)

    except Exception as e:
        print(f'  ⚠️ S2 extraction failed for ({lat:.4f}, {lon:.4f}): {e}')

    # ── Sentinel-1 SAR ────────────────────────────────────────────────────
    try:
        s1 = (
            ee.ImageCollection('COPERNICUS/S1_GRD')
            .filterDate(start_str, end_str)
            .filterBounds(aoi)
            .filter(ee.Filter.eq('instrumentMode', 'IW'))
            .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
            .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))
            .select(['VV', 'VH'])
        )

        n_s1 = s1.size().getInfo()
        if n_s1 > 0:
            def _extract_sar(img):
                stats = img.reduceRegion(
                    reducer=ee.Reducer.mean(), geometry=aoi, scale=10, maxPixels=1e6
                )
                return ee.Feature(None, {'vv': stats.get('VV'), 'vh': stats.get('VH')})

            sar_fc = s1.map(_extract_sar).getInfo()
            vv_vals, vh_vals = [], []
            for f in sar_fc.get('features', []):
                vv = f['properties'].get('vv')
                vh = f['properties'].get('vh')
                if vv is not None:
                    vv_vals.append(float(vv))
                if vh is not None:
                    vh_vals.append(float(vh))

            if vv_vals:
                features['vv_mean'] = float(np.mean(vv_vals))
                features['vv_std']  = float(np.std(vv_vals))
            if vh_vals:
                features['vh_mean'] = float(np.mean(vh_vals))
                features['vh_std']  = float(np.std(vh_vals))
            if vv_vals and vh_vals and len(vv_vals) == len(vh_vals):
                vv_arr = np.array(vv_vals)
                vh_arr = np.array(vh_vals)
                features['vh_vv_ratio'] = float(np.mean(vh_arr / (vv_arr + 1e-8)))

    except Exception as e:
        print(f'  ⚠️ S1 extraction failed for ({lat:.4f}, {lon:.4f}): {e}')

    return features


print('✅ Feature extraction function defined.')
print('   Schema: 17 features matching training_features.csv exactly.')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Run Feature Extraction on All Collected Points
# ═══════════════════════════════════════════════════════════════════════════

if not collected_points:
    print('⚠️  No points collected yet. Go back to Section 3 or 4 to add points.')
else:
    print(f'🔄 Extracting 17 features for {len(collected_points)} points...\n')

    FEATURE_COLS = [
        'blue_mean', 'green_mean', 'red_mean', 'nir_mean', 'swir1_mean',
        'ndvi_mean', 'ndvi_min', 'ndvi_max', 'ndvi_std', 'ndvi_range', 'ndvi_slope',
        'ndwi_mean',
        'vv_mean', 'vv_std', 'vh_mean', 'vh_std', 'vh_vv_ratio',
    ]

    featurized_records = []
    for i, pt in enumerate(collected_points):
        print(f'  [{i+1}/{len(collected_points)}] {pt["crop"]} @ {pt["lat"]:.4f}°N, {pt["lon"]:.4f}°E ({pt["district"]})...')
        feats = extract_17_features(pt['lat'], pt['lon'], pt['date'])

        record = {
            'field_id': f'MP_UP_{i:05d}',
            'crop': pt['crop'],
            'state': pt.get('state', SELECTED_STATE),
            'lat': pt['lat'],
            'lon': pt['lon'],
        }
        for col in FEATURE_COLS:
            record[col] = round(feats.get(col, 0.0), 6)
        record['source'] = pt.get('source', 'manual_click')

        featurized_records.append(record)
        print(f'    ✅ NDVI={record["ndvi_mean"]:.4f}, VV={record["vv_mean"]:.2f} dB')

    featurized_df = pd.DataFrame(featurized_records)
    print(f'\n🎉 Feature extraction complete! {len(featurized_df)} records with 17 features.')
    display(featurized_df)

---
## 6. Export

The CSV uses the **exact same column schema** as `data/ground_truth/training_features.csv` — it merges directly into `load_merged_ground_truth()` with zero schema mapping.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Export to CSV
# ═══════════════════════════════════════════════════════════════════════════

OUTPUT_DIR = Path('data/ground_truth')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH = OUTPUT_DIR / 'mp_up_manual_labels.csv'

EXPORT_COLS = [
    'field_id', 'crop', 'state', 'lat', 'lon',
    'blue_mean', 'green_mean', 'red_mean', 'nir_mean', 'swir1_mean',
    'ndvi_mean', 'ndvi_min', 'ndvi_max', 'ndvi_std', 'ndvi_range', 'ndvi_slope',
    'ndwi_mean',
    'vv_mean', 'vv_std', 'vh_mean', 'vh_std', 'vh_vv_ratio',
    'source',
]

if 'featurized_df' not in dir() or featurized_df.empty:
    print('⚠️  No featurized data to export. Run Section 5 first.')
else:
    for col in EXPORT_COLS:
        if col not in featurized_df.columns:
            featurized_df[col] = 0.0 if col not in ['field_id', 'crop', 'state', 'source'] else ''

    export_df = featurized_df[EXPORT_COLS]

    if OUTPUT_PATH.exists():
        existing = pd.read_csv(OUTPUT_PATH)
        combined = pd.concat([existing, export_df], ignore_index=True)
        combined = combined.drop_duplicates(subset=['field_id'], keep='last')
        combined.to_csv(OUTPUT_PATH, index=False)
        print(f'📥 Appended to existing file. Total: {len(combined)} records.')
    else:
        export_df.to_csv(OUTPUT_PATH, index=False)
        print(f'📥 Saved {len(export_df)} records to:')

    print(f'   📄 {OUTPUT_PATH}')
    print(f'\n   This file merges directly into load_merged_ground_truth().')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Summary — Point Counts per District & per Crop (for report screenshots)
# ═══════════════════════════════════════════════════════════════════════════

if 'featurized_df' in dir() and not featurized_df.empty:
    print('═' * 60)
    print('  📊  GROUND-TRUTH COLLECTION SUMMARY')
    print('═' * 60)
    print(f'\n  Total labeled points:  {len(featurized_df)}')
    print(f'  State(s):              {state_label}')
    print(f'  Season:                {SEASON} {SEASON_YEAR}')
    print(f'  Output file:           {OUTPUT_PATH}')

    print('\n' + '─' * 60)
    print('  🌾  Points by Crop Type')
    print('─' * 60)
    crop_counts = featurized_df['crop'].value_counts()
    for crop, count in crop_counts.items():
        bar = '█' * count
        print(f'  {crop:<14s}  {count:>4d}  {bar}')

    if 'state' in featurized_df.columns:
        print('\n' + '─' * 60)
        print('  🗺️  Points by State')
        print('─' * 60)
        state_counts = featurized_df['state'].value_counts()
        for st, count in state_counts.items():
            bar = '█' * count
            print(f'  {st:<20s}  {count:>4d}  {bar}')

    if collected_points:
        labels_df = pd.DataFrame(collected_points)
        if 'district' in labels_df.columns:
            non_empty = labels_df[labels_df['district'].str.strip() != '']
            if not non_empty.empty:
                print('\n' + '─' * 60)
                print('  📍  Points by District')
                print('─' * 60)
                district_counts = non_empty['district'].value_counts()
                for dist, count in district_counts.items():
                    bar = '█' * count
                    print(f'  {dist:<20s}  {count:>4d}  {bar}')

    print('\n' + '─' * 60)
    print('  🔗  Points by Source')
    print('─' * 60)
    source_counts = featurized_df['source'].value_counts()
    for src, count in source_counts.items():
        bar = '█' * count
        print(f'  {src:<16s}  {count:>4d}  {bar}')

    if featurized_df['state'].nunique() > 1:
        print('\n' + '─' * 60)
        print('  📊  Crop × State Cross-Tabulation')
        print('─' * 60)
        cross = pd.crosstab(featurized_df['crop'], featurized_df['state'], margins=True)
        display(cross)

    print('\n' + '═' * 60)
    print('  ✅  Report-ready summary complete.')
    print('═' * 60)
else:
    print('⚠️  No data to summarize. Complete Sections 3-5 first.')

---
## 7. Quick Reference

### Workflow summary

```
┌─────────────────────────────────────────────────────────┐
│  1. Setup & authenticate GEE                            │
│  2. Select state (MP / UP / Both) + season (Rabi/Kharif)│
│     → Map is LOCKED to state bounds (can't pan outside) │
│     → Imagery clipped to exact polygon (not bbox)       │
│  3. Click points on the map → tag crop/district/date    │
│     (or import KML/GeoJSON polygons)                    │
│  4. Run 17-feature extraction from S2 + S1              │
│  5. Export mp_up_manual_labels.csv                       │
│  6. Merge into training pipeline via                    │
│     load_merged_ground_truth() — zero schema changes    │
└─────────────────────────────────────────────────────────┘
```

### Crop classes (10)

| Class | Typical Season | Key States |
|-------|---------------|------------|
| Wheat | Rabi | MP, UP |
| Rice | Kharif | UP, MP |
| Mustard | Rabi | UP, MP |
| Soybean | Kharif | MP |
| Cotton | Kharif | MP |
| Sugarcane | Both | UP |
| Potato | Rabi | UP |
| Lentil | Rabi | MP, UP |
| Maize | Kharif | UP, MP |
| Gram | Rabi | MP |